<a href="https://colab.research.google.com/github/Jalilnkh/PyTorch-with-Examples-2024/blob/parts/en_azb_mt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#Checking if GPU is running or not

!nvidia-smi

Thu Dec 12 17:34:30 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   38C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [5]:
!pip install datasets transformers[sentencepiece] sacrebleu -q

In [6]:
import os
import sys
import transformers
import tensorflow as tf
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import TFAutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import AdamWeightDecay
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

In [7]:
model_checkpoint = "Helsinki-NLP/opus-mt-en-az"

In [19]:
raw_datasets = load_dataset("Kartal-Ol/en-azb-548k")

en-azb-548k.parquet:   0%|          | 0.00/87.9M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [20]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 548900
    })
})

In [21]:
raw_datasets['train'][1]

{'translation': {'azb': '( سیزین اؤزونوزه سلام اوْلماسین ! )',
  'en': 'No welcome for you !'}}

In [22]:
raw_datasets['train'][100000]

{'translation': {'azb': 'یئهووا هابیلین قُربانینی نه\u200cیه گؤره تقدیر ائتدی ؟',
  'en': 'Why did Jehovah look with favor on Abel ’ s offering ?'}}

In [23]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/451k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/470k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/598k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [24]:
tokenizer("Hello, this is a sentence!")

{'input_ids': [1236, 436, 6, 106, 18, 16, 9415, 186, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [25]:
tokenizer(["Hello, this is a sentence!", "This is another sentence."])

{'input_ids': [[1236, 436, 6, 106, 18, 16, 9415, 186, 0], [288, 18, 455, 9415, 5, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}

In [26]:
with tokenizer.as_target_tokenizer():
    print(tokenizer(['( سیزین اؤزونوزه سلام اوْلماسین !']))

{'input_ids': [[7, 28, 1, 23200, 28, 1, 23200, 1, 28, 1, 28, 1, 23200, 14, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [27]:
max_input_length = 128
max_target_length = 128

source_lang = "en"
target_lang = "azb"


def preprocess_function(examples):
    inputs = [ex[source_lang] for ex in examples["translation"]]
    targets = [ex[target_lang] for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # Setup the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [28]:
preprocess_function(raw_datasets["train"][:2])

{'input_ids': [[10946, 68, 1010, 328, 54, 8264, 328, 12883, 37, 2740, 257, 21234, 20344, 56, 813, 2, 12883, 37, 2740, 257, 21234, 970, 1295, 56, 187, 301, 29, 2856, 14, 800, 90, 97, 221, 16, 7344, 11, 2223, 155, 62, 3277, 3, 0], [563, 9068, 30, 17, 14, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]], 'labels': [[28, 1, 23199, 1, 23200, 28, 1, 28, 1, 23199, 1, 23200, 1, 23200, 7, 28, 1, 28, 1, 28, 1, 28, 1, 23199, 1, 8, 28, 1, 23200, 1, 28, 1, 28, 1, 26, 28, 89, 28, 1, 28, 1, 28, 1, 28, 23199, 1, 23200, 28, 1, 23200, 1, 23200, 28, 1, 23200, 1, 23200, 28, 1, 23200, 1, 23200, 28, 1, 23199, 28, 1, 28, 1, 3, 0], [7, 28, 1, 23200, 28, 1, 23200, 1, 28, 1, 28, 1, 23200, 14, 8, 0]]}

In [29]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/548900 [00:00<?, ? examples/s]

In [30]:
model = TFAutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

tf_model.h5:   0%|          | 0.00/227M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFMarianMTModel.

All the layers of TFMarianMTModel were initialized from the model checkpoint at Helsinki-NLP/opus-mt-en-az.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMarianMTModel for predictions without further training.


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [31]:
batch_size = 16
learning_rate = 2e-5
weight_decay = 0.01
num_train_epochs = 5

In [32]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, return_tensors="tf")

In [33]:
generation_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, return_tensors="tf", pad_to_multiple_of=128)

In [35]:
train_dataset = model.prepare_tf_dataset(
    tokenized_datasets["train"],
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator,
)

In [ ]:
validation_dataset = model.prepare_tf_dataset(
    tokenized_datasets["validation"],
    batch_size=batch_size,
    shuffle=False,
    collate_fn=data_collator,
)